In [2]:
# 代码作用：处理原糖生产公司月累计报数据，从源表读取数据后进行去重处理，
# 补充数据来源和跑批时间等审计字段，最终全量覆盖写入目标表
# 日期：20250826
# 作者：yushumeng
import pandas as pd
from datetime import datetime
from sqlalchemy import create_engine
from pyspark.sql.types import StructType, StructField, StringType, TimestampType
from pyspark.sql.functions import current_timestamp
from pyspark.sql import SparkSession
import warnings
warnings.filterwarnings("ignore")

print("======================spark开启=====================================")
spark = SparkSession.builder \
    .appName("lims_dwd") \
    .enableHiveSupport() \
    .getOrCreate()

# 配置参数
source_table = "ods.ods_lims_bi_ytribaogs_hx_ylj_f_1h" 
target_table = "dwd.dwd_lims_sugar_production_company_month_f_1d"
dedup_keys = ["id"]
exclude_audit_fields = ['source_flg', 'dw_update_time']  # 排除源表中已有的审计字段

# 读取源表Schema，过滤审计字段
source_schema = spark.table(source_table).schema
source_fields = [f.name for f in source_schema.fields if f.name not in exclude_audit_fields]

# 生成查询SQL
select_fields = ", ".join(source_fields)
lims_sql = f"""
SELECT 
    {select_fields},
    'lims' AS source_flg,
    current_timestamp() AS dw_update_time
FROM {source_table}
"""

# 执行查询
lims_df = spark.sql(lims_sql)
original_count = lims_df.count()
print(f"源表原始数据量：{original_count} 条")

# 去重处理
dedup_df = lims_df.dropDuplicates(dedup_keys)
dedup_count = dedup_df.count()
print(f"去重后数据量：{dedup_count} 条，去除重复记录：{original_count - dedup_count} 条")

# 构建目标Schema
source_struct_fields = [
    StructField(field.name, field.dataType, True,
               metadata={"comment": field.metadata.get("comment", "无备注")})
    for field in source_schema if field.name not in exclude_audit_fields
]

audit_struct_fields = [
    StructField("source_flg", StringType(), True, metadata={"comment": "数据来源标识"}),
    StructField("dw_update_time", TimestampType(), True, metadata={"comment": "ETL跑批时间"})
]

target_schema = StructType(source_struct_fields + audit_struct_fields)

# 应用Schema并写入
spark_df = spark.createDataFrame(dedup_df.rdd, schema=target_schema)
print(f"执行全量覆盖写入表：{target_table}")
spark_df.write.mode("overwrite") \
    .option("comment", "原糖生产公司月累计报") \
    .saveAsTable(target_table)  

spark.stop()
print("======================spark关闭=====================================")


======================spark开启=====================================
源表原始数据量：3475 条
去重后数据量：3475 条，去除重复记录：0 条
执行全量覆盖写入表：dwd.dwd_lims_sugar_production_company_month_f_1d
======================spark关闭=====================================


In [ ]:
# 代码作用：将原糖日报填报数据表与原糖生产公司日报表通过 id 进行左连接，形成  原糖生产公司日报  的主题表
# 对两表重复字段添加左表前缀处理，保留原字段注释并补充审计字段，最终写入目标表
# 日期：20250826
# 作者：yushumeng
import pandas as pd
from datetime import datetime
from sqlalchemy import create_engine
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, IntegerType, DecimalType
from pyspark.sql.functions import current_timestamp
from pyspark.sql import SparkSession
import warnings
warnings.filterwarnings("ignore")

print("======================spark开启=====================================")
spark = SparkSession.builder \
    .appName("lims_dwr_sugar_daily_production")\
    .enableHiveSupport() \
    .getOrCreate()

# 核心配置
exclude_audit_fields = ['source_flg', 'dw_update_time']  # 需排除的审计字段
join_key = 'id'  # 关联键（全小写）
left_prefix = 'tb_'  # 左表重复字段前缀

# 源表定义
tb_table = "dwd.dwd_lims_raw_daily_report_detail_f_1d" 
fac_table = "dwd.dwd_lims_sugar_production_company_daily_f_1d"  

# 获取字段及Schema
tb_schema = spark.table(tb_table).schema
tb_all_fields = [f.name for f in tb_schema.fields] 
fac_schema = spark.table(fac_table).schema
fac_all_fields = [f.name for f in fac_schema.fields] 

# 检测重复字段（排除关联键和审计字段）
business_duplicate_fields = list(
    (set(tb_all_fields) & set(fac_all_fields)) 
    - set(exclude_audit_fields) 
    - set([join_key])
)
print(f"重复字段（左表加前缀{left_prefix}，右表原名）：{business_duplicate_fields}")

# 左表字段处理：非重复字段保留原名，重复字段加前缀
tb_non_duplicate = [f for f in tb_all_fields if f not in business_duplicate_fields and f not in exclude_audit_fields]
tb_sql_fields = [f"tb.{f} AS {f}" for f in tb_non_duplicate]  # 非重复字段
tb_sql_fields += [f"tb.{f} AS {left_prefix}{f}" for f in business_duplicate_fields]  # 重复字段加前缀
tb_sql_fields_str = ", ".join(tb_sql_fields)

# 右表字段处理：排除关联键和审计字段
fac_keep_fields = [f for f in fac_all_fields if f != join_key and f not in exclude_audit_fields]
fac_sql_fields_str = ", ".join([f"fac.{f}" for f in fac_keep_fields])

# 拼接查询SQL
lims_sql = f"""
SELECT 
    {tb_sql_fields_str}, 
    {fac_sql_fields_str},  
    'lims' AS source_flg,  
    current_timestamp() AS dw_update_time  
FROM {tb_table} tb  
LEFT JOIN {fac_table} fac  
ON tb.{join_key} = fac.{join_key};
"""

# 执行查询
lims_df = spark.sql(lims_sql)

# 校验重复字段
df_columns = [col for col in lims_df.columns]
duplicate_cols = [col for col in df_columns if df_columns.count(col) > 1]
if duplicate_cols:
    raise ValueError(f"仍有重复字段：{duplicate_cols}")
else:
    print(f"左连接成功，字段数：{len(lims_df.columns)}")

# 构建完整Schema
tb_non_dup_struct = [
    StructField(f, tb_schema[tb_schema.fieldNames().index(f)].dataType, True,
               metadata={"comment": tb_schema[tb_schema.fieldNames().index(f)].metadata.get('comment', '无备注')})
    for f in tb_non_duplicate
]

tb_dup_struct = [
    StructField(f"{left_prefix}{f}", tb_schema[tb_schema.fieldNames().index(f)].dataType, True,
               metadata={"comment": tb_schema[tb_schema.fieldNames().index(f)].metadata.get('comment', '无备注')})
    for f in business_duplicate_fields
]

fac_struct = [
    StructField(f, fac_schema[fac_schema.fieldNames().index(f)].dataType, True,
               metadata={"comment": fac_schema[fac_schema.fieldNames().index(f)].metadata.get('comment', '无备注')})
    for f in fac_keep_fields
]

audit_struct = [
    StructField("source_flg", StringType(), True, metadata={"comment": "来源库"}),
    StructField("dw_update_time", TimestampType(), True, metadata={"comment": "ETL跑批时间"})
]

full_schema = StructType(tb_non_dup_struct + tb_dup_struct + fac_struct + audit_struct)

# 应用Schema
spark_df = spark.createDataFrame(lims_df.rdd, schema=full_schema)

# 写入目标表
target_table = "dwr.dwr_lims_sugar_daily_production_company_f_1d"
print(f"写入表：{target_table}")
spark_df.write.mode("overwrite") \
    .option("comment", "原糖生产公司日报表") \
    .saveAsTable(target_table)  

# 打印结果
record_count = spark_df.count()
col_count = len(spark_df.columns)
print(f"写入成功：{record_count} 条，{col_count} 个字段")

spark.stop()
print("======================spark关闭=====================================")
